In [ ]:
import cv2
import os
import glob
import random
import shutil
import numpy as np
from tqdm.notebook import tqdm

# ================= CONFIGURATION =================
dataset_path = r"D:\Plant_Identification\Dataset\Original_Merge"
sample_folder = r"../Datasets/Sampling"
TARGET_COUNT = 750
# =================================================

def augment_image(image):
    rows, cols, _ = image.shape
    choice = random.choice(['rotate', 'flip', 'noise', 'brightness', 'zoom'])

    if choice == 'rotate':
        angle = random.randint(-25, 25)
        M = cv2.getRotationMatrix2D((cols/2, rows/2), angle, 1)
        return cv2.warpAffine(image, M, (cols, rows))
    elif choice == 'flip':
        return cv2.flip(image, random.choice([0, 1, -1]))
    elif choice == 'noise':
        noise = np.random.normal(0, 15, (rows, cols, 3)).astype(np.uint8)
        return cv2.add(image, noise)
    elif choice == 'brightness':
        value = random.randint(-40, 40)
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv)
        v = cv2.add(v, value)
        final_hsv = cv2.merge((h, s, v))
        return cv2.cvtColor(final_hsv, cv2.COLOR_HSV2BGR)
    elif choice == 'zoom':
        margin = random.randint(10, 50)
        if rows - margin > 0 and cols - margin > 0:
            crop = image[margin:rows-margin, margin:cols-margin]
            return cv2.resize(crop, (cols, rows))
        else:
            return image
    return image 

def create_balanced_dataset():
    if not os.path.exists(dataset_path):
        print(f"❌ ERROR: Source folder not found at {dataset_path}")
        return

    folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
    
    print(f"📊 Balancing dataset to exactly {TARGET_COUNT} images per class...")
    print(f"📂 Output Location: {sample_folder}\n")

    for folder in tqdm(folders, desc="Overall Progress"):
        source_class_path = os.path.join(dataset_path, folder)
        target_class_path = os.path.join(sample_folder, folder)
        
        # Create destination folder
        os.makedirs(target_class_path, exist_ok=True)
        
        extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
        images = []
        for ext in extensions:
            images.extend(glob.glob(os.path.join(source_class_path, ext)))
            
        current_count = len(images)
        
        if current_count == 0:
            print(f"⚠️ Warning: No images found in '{folder}'. Skipping.")
            continue
        
        # 2. DECIDE WHAT TO COPY
        if current_count > TARGET_COUNT:
            # Too many files? Randomly pick 750
            images_to_copy = random.sample(images, TARGET_COUNT)
            needed = 0
        else:
            # Too few files? Copy all of them
            images_to_copy = images
            needed = TARGET_COUNT - current_count

        # 3. COPY ORIGINAL IMAGES (With debug print)
        # print(f"   -> Copying {len(images_to_copy)} originals for {folder}...") 
        for img_path in images_to_copy:
            try:
                filename = os.path.basename(img_path)
                shutil.copy2(img_path, os.path.join(target_class_path, filename))
            except Exception as e:
                print(f"   ❌ Error copying {filename}: {e}")

        # 4. AUGMENT TO FILL GAP
        if needed > 0:
            desc_str = f"Augmenting {folder}"
            for i in tqdm(range(needed), desc=desc_str, leave=False):
                # Pick random original image to augment
                rand_img_path = random.choice(images)
                original = cv2.imread(rand_img_path)
                
                if original is None: continue 
                    
                augmented = augment_image(original)
                
                # Save to NEW folder
                new_name = f"aug_{i}_{os.path.basename(rand_img_path)}"
                save_path = os.path.join(target_class_path, new_name)
                cv2.imwrite(save_path, augmented)

    print(f"\n✅ Done! Check your folder: {sample_folder}")

# Run the function
create_balanced_dataset()